In [ ]:
import pandas as pd
import networkx as nx
import osmnx as ox
import numpy as np
import random
import sys
from pathlib import Path

# Import scoring module for canonical cost formula and weights
sys.path.insert(0, str(Path("../src").resolve()))
from scoring import safety_cost, composite_score, get_weights, get_buffer_size

# 1. LOAD DATA — relative paths so this runs on any machine
GRAPH_PATH = '../data/processed/sd_walk_graph.graphml'
SCORES_PATH = '../data/processed/edge_scores_infrastructure.csv'

print("Loading GraphML file... (This may take a minute for large files)")
G = ox.load_graphml(GRAPH_PATH)

print("Loading Scores CSV...")
df_scores = pd.read_csv(SCORES_PATH)
df_scores.set_index(['u', 'v', 'key'], inplace=True)

print(f"Graph edges: {G.number_of_edges():,}")
print(f"Score rows:  {len(df_scores):,}")

# 2. DEFINE ROUTING PROFILES (matches safety-score-edge.ipynb)
profiles = {
    'Safety_Day': {
        'crime_col': 'crime_score_medium_day',
        'weights': get_weights(is_night=False),
    },
    'Safety_Night': {
        'crime_col': 'crime_score_medium_night',
        'weights': get_weights(is_night=True),
    },
    'Standard_Shortest': None,  # pure length
}

def calculate_cost(edge_data, profile):
    """Calculate routing cost for an edge using the canonical formula."""
    if profile is None:
        return edge_data.get('length', 1.0)
    w = profile['weights']
    crime = edge_data.get('crime', 0.5)
    walk = edge_data.get('walk', 0.5)
    infra = edge_data.get('infra', 0.5)
    score = w['crime'] * crime + w['walk'] * walk + w['infra'] * infra
    return safety_cost(edge_data.get('length', 1.0), score)

# 3. ATTACH SCORES TO GRAPH
print("Merging scores into graph edges and calculating costs...")
for u, v, k, data in G.edges(keys=True, data=True):
    try:
        row = df_scores.loc[(u, v, k)]
        data['crime_day'] = row['crime_score_medium_day']
        data['crime_night'] = row['crime_score_medium_night']
        data['walk'] = row['walk_score']
        data['infra'] = row['infrastructure_score']
    except KeyError:
        data['crime_day'] = 0.5
        data['crime_night'] = 0.5
        data['walk'] = 0.5
        data['infra'] = 0.5

    for p_name, p_config in profiles.items():
        if p_config is None:
            data[f'cost_{p_name}'] = data.get('length', 1.0)
        else:
            edge_for_cost = dict(data)
            edge_for_cost['crime'] = data['crime_day'] if 'Day' in p_name else data['crime_night']
            data[f'cost_{p_name}'] = calculate_cost(edge_for_cost, p_config)

# 4. RUN TEST ROUTES
nodes = list(G.nodes())
random.seed(42)

source, target = None, None
print("Searching for a valid connected route...")
for _ in range(200):
    s, t = random.sample(nodes, 2)
    if nx.has_path(G, s, t):
        source, target = s, t
        break

if not source:
    print("Error: Could not find connected nodes. Check your GraphML connectivity.")
else:
    print(f"Testing route from {source} to {target}\n")
    results = []

    for profile_name in profiles.keys():
        weight_attr = f'cost_{profile_name}'
        try:
            path = nx.shortest_path(G, source, target, weight=weight_attr)

            path_edge_values = []
            for u, v in zip(path[:-1], path[1:]):
                edges = G.get_edge_data(u, v)
                best_edge = min(edges.values(), key=lambda x: x.get(weight_attr, float('inf')))
                path_edge_values.append(best_edge)

            stats = {
                'Profile': profile_name,
                'Steps': len(path) - 1,
                'Total_Dist (m)': sum(e.get('length', 0) for e in path_edge_values),
                'Avg_Crime_Day': np.mean([e.get('crime_day', 0) for e in path_edge_values]),
                'Avg_Crime_Night': np.mean([e.get('crime_night', 0) for e in path_edge_values]),
                'Avg_Walk': np.mean([e.get('walk', 0) for e in path_edge_values]),
                'Avg_Infra': np.mean([e.get('infra', 0) for e in path_edge_values]),
            }
            results.append(stats)

        except nx.NetworkXNoPath:
            print(f"No path found for {profile_name}")

    # 5. DISPLAY VALIDATION TABLE
    df_results = pd.DataFrame(results)
    print("--- VALIDATION RANKINGS ---")
    print(df_results.sort_values('Avg_Crime_Day', ascending=False).to_string(index=False))

    # Expected:
    # 1. Safety_Day should have the highest Avg_Crime_Day score.
    # 2. Safety_Night should have the highest Avg_Infra score.
    # 3. Standard_Shortest should have the lowest Total_Dist.


## Testing initial scoring on sample routes to compare outputs and validate whether rankings are reasonable and interpretable.

- Standard_Shortest has lowest total distance and lowest safety scores (Day and Night)
- Safety_Night has highest Avg_light
- Safety_Day has best Safety score for day and night
- Shortest routes has less light than routes in daytime